# Research RagChat: AI-Powered Research Paper Chatbot

A simple RAG pipeline for asking questions about a research paper. The pipeline loads a PDF, splits it into chunks, creates embeddings, stores them in FAISS, retrieves relevant chunks, and uses an OpenAI model to generate an answer.

## 1. Install libraries
Install the packages required for PDF processing, LangChain, embeddings, and FAISS.

In [ ]:
!pip -q install -U langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu pypdf

## 2. API key
The API key is entered securely instead of being written directly in the notebook.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Imports
Import the components used in the RAG pipeline.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS

## 4. Upload research paper
Upload the PDF that you want to use as the knowledge source.

In [ ]:
from google.colab import files

uploaded = files.upload()
pdf_path = next(iter(uploaded.keys()))

print("Uploaded:", pdf_path)

## 5. Load PDF
Extract the text from the research paper while keeping page information as metadata.

In [ ]:
loader = PyPDFLoader(pdf_path)
documents = loader.load()

print("Number of pages:", len(documents))

## 6. Inspect extracted text
Check that the PDF was extracted correctly before continuing.

In [ ]:
print(documents[0].page_content[:2000])

## 7. Split documents
Large documents are divided into smaller overlapping chunks so relevant information can be retrieved efficiently.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)

print("Number of chunks:", len(chunks))

## 8. Inspect chunks
Check a chunk and its metadata, especially the source page.

In [ ]:
print(chunks[0].page_content)
print("\nMetadata:", chunks[0].metadata)

## 9. Create embeddings
Convert each text chunk into a numerical vector that represents its semantic meaning.

In [ ]:
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

print("Embedding model ready.")

## 10. Create FAISS vector store
Store the chunk embeddings in FAISS so that similar content can be retrieved quickly.

In [ ]:
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("FAISS vector store created.")

## 11. Create retriever
Retrieve the four most relevant chunks for each question.

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

print("Retriever ready.")

## 12. Create LLM
Use GPT-4o-mini to generate answers from the retrieved research-paper context.

In [ ]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

print("LLM ready.")

## 13. RAG prompt
The prompt tells the model to use only the retrieved context and avoid unsupported answers.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are an AI research paper assistant.

Answer the question using ONLY the context provided below.
If the answer cannot be found in the context, say:
"I don't know based on the provided research paper."
Do not make up information.

Context:
{context}

Question:
{question}

Answer:
""")

## 14. RAG function
Retrieve relevant chunks, place them into the prompt, and send the result to the LLM.

In [ ]:
def rag_query(question):
    retrieved_docs = retriever.invoke(question)

    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    messages = prompt.invoke({
        "context": context,
        "question": question
    })

    response = llm.invoke(messages)

    return response.content, retrieved_docs

## 15. Ask a question
Ask a question that is answered by the research paper.

In [ ]:
question = "What is Retrieval Augmented Generation?"

answer, sources = rag_query(question)
print(answer)

## 16. Display sources
Show the pages used to generate the answer so the result can be checked against the paper.

In [ ]:
print("ANSWER:\n")
print(answer)

print("\nSOURCES:\n")
for doc in sources:
    page = doc.metadata.get("page")
    print(f"- Page {page + 1}" if page is not None else "- Page information unavailable")

## 17. Test unsupported information
Ask something unrelated to the paper. The system should avoid making up an answer.

In [ ]:
question = "Who is Sachin Tendulkar?"

answer, sources = rag_query(question)
print(answer)

## RAG Architecture

```text
Research PDF
     ↓
PDF Loader
     ↓
Text Chunking
     ↓
OpenAI Embeddings
     ↓
FAISS Vector Store
     ↓
Retriever
     ↓
Relevant Chunks
     ↓
RAG Prompt
     ↓
GPT-4o-mini
     ↓
Answer + Source Pages
```